# 01 — Data Pipeline: Olist → `master_df.parquet`

**Owner:** Vybhav
**Purpose:** Merge 9 Olist CSVs → single wide DataFrame → filter SP → save parquet  
**Output:** `data/master_df.parquet` (~40K rows)

---

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

RAW_DIR = Path("../data/raw")
print("Files in data/raw:")
for f in sorted(RAW_DIR.glob("*.csv")):
    print(f"  {f.name:>50s}  {f.stat().st_size / 1e6:.1f} MB")

Files in data/raw:
                         olist_customers_dataset.csv  9.0 MB
                       olist_geolocation_dataset.csv  61.3 MB
                       olist_order_items_dataset.csv  15.4 MB
                    olist_order_payments_dataset.csv  5.8 MB
                     olist_order_reviews_dataset.csv  14.5 MB
                            olist_orders_dataset.csv  17.7 MB
                          olist_products_dataset.csv  2.4 MB
                           olist_sellers_dataset.csv  0.2 MB
               product_category_name_translation.csv  0.0 MB


## 1. Load all 9 Olist CSVs

In [2]:
orders    = pd.read_csv(RAW_DIR / "olist_orders_dataset.csv")
items     = pd.read_csv(RAW_DIR / "olist_order_items_dataset.csv")
products  = pd.read_csv(RAW_DIR / "olist_products_dataset.csv")
customers = pd.read_csv(RAW_DIR / "olist_customers_dataset.csv")
sellers   = pd.read_csv(RAW_DIR / "olist_sellers_dataset.csv")
geo       = pd.read_csv(RAW_DIR / "olist_geolocation_dataset.csv")
reviews   = pd.read_csv(RAW_DIR / "olist_order_reviews_dataset.csv")
payments  = pd.read_csv(RAW_DIR / "olist_order_payments_dataset.csv")
transl    = pd.read_csv(RAW_DIR / "product_category_name_translation.csv")

tables = {
    "orders": orders, "items": items, "products": products,
    "customers": customers, "sellers": sellers, "geo": geo,
    "reviews": reviews, "payments": payments, "translation": transl,
}
for name, df in tables.items():
    print(f"{name:>15s}  {df.shape[0]:>9,} rows × {df.shape[1]:>3} cols")

         orders     99,441 rows ×   8 cols
          items    112,650 rows ×   7 cols
       products     32,951 rows ×   9 cols
      customers     99,441 rows ×   5 cols
        sellers      3,095 rows ×   4 cols
            geo  1,000,163 rows ×   5 cols
        reviews     99,224 rows ×   7 cols
       payments    103,886 rows ×   5 cols
    translation         71 rows ×   2 cols


## 2. Schema peek — key columns

In [3]:
print("=== orders ===")
print(orders.dtypes.to_string())
print(f"\norder_status values: {orders['order_status'].value_counts().to_dict()}")

=== orders ===
order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str

order_status values: {'delivered': 96478, 'shipped': 1107, 'canceled': 625, 'unavailable': 609, 'invoiced': 314, 'processing': 301, 'created': 5, 'approved': 2}


## 3. Geolocation — deduplicate zips to median lat/lon

In [4]:
print(f"Raw geolocation rows: {len(geo):,}")
print(f"Unique zip prefixes:  {geo['geolocation_zip_code_prefix'].nunique():,}")

geo_median = (
    geo.groupby("geolocation_zip_code_prefix", as_index=False)
    .agg(lat=("geolocation_lat", "median"), lon=("geolocation_lng", "median"))
)
print(f"After dedup:          {len(geo_median):,} rows")
geo_median.head()

Raw geolocation rows: 1,000,163
Unique zip prefixes:  19,015
After dedup:          19,015 rows


,geolocation_zip_code_prefix,lat,lon
0,1001,-23.550381,-46.634027
1,1002,-23.548551,-46.635072
2,1003,-23.548977,-46.635313
3,1004,-23.549535,-46.634771
4,1005,-23.549612,-46.636532


## 4. Merge chain

In [5]:
# --- Customers + geo ---
customers_geo = customers.merge(
    geo_median.rename(columns={"lat": "customer_lat", "lon": "customer_lon"}),
    left_on="customer_zip_code_prefix",
    right_on="geolocation_zip_code_prefix",
    how="left",
).drop(columns=["geolocation_zip_code_prefix"], errors="ignore")
print(f"customers_geo: {len(customers_geo):,} rows, "
      f"null coords: {customers_geo['customer_lat'].isna().sum():,}")

# --- Sellers + geo ---
sellers_geo = sellers.merge(
    geo_median.rename(columns={"lat": "seller_lat", "lon": "seller_lon"}),
    left_on="seller_zip_code_prefix",
    right_on="geolocation_zip_code_prefix",
    how="left",
).drop(columns=["geolocation_zip_code_prefix"], errors="ignore")
print(f"sellers_geo:   {len(sellers_geo):,} rows, "
      f"null coords: {sellers_geo['seller_lat'].isna().sum():,}")

customers_geo: 99,441 rows, null coords: 278
sellers_geo:   3,095 rows, null coords: 7


In [6]:
# --- Products + English names ---
products_en = products.merge(transl, on="product_category_name", how="left")
print(f"products_en: {len(products_en):,} rows, "
      f"translated: {products_en['product_category_name_english'].notna().sum():,}")

# --- Items + products + sellers ---
items_merged = (
    items
    .merge(
        products_en[["product_id", "product_category_name_english",
                      "product_weight_g", "product_length_cm",
                      "product_height_cm", "product_width_cm"]],
        on="product_id", how="left",
    )
    .merge(
        sellers_geo[["seller_id", "seller_lat", "seller_lon",
                     "seller_state", "seller_city", "seller_zip_code_prefix"]],
        on="seller_id", how="left",
    )
)
print(f"items_merged: {len(items_merged):,} rows")

# Aggregate per order (sum price/freight/weight, keep first category/seller)
items_agg = items_merged.groupby("order_id", as_index=False).agg(
    price=("price", "sum"),
    freight_value=("freight_value", "sum"),
    product_weight_g=("product_weight_g", "sum"),
    n_items=("product_id", "count"),
    product_id=("product_id", "first"),
    product_category_name_english=("product_category_name_english", "first"),
    seller_lat=("seller_lat", "first"),
    seller_lon=("seller_lon", "first"),
    seller_state=("seller_state", "first"),
    seller_city=("seller_city", "first"),
    seller_zip_code_prefix=("seller_zip_code_prefix", "first"),
)
print(f"items_agg (order-level): {len(items_agg):,} rows")

products_en: 32,951 rows, translated: 32,328
items_merged: 112,650 rows
items_agg (order-level): 98,666 rows


In [7]:
# --- Reviews (latest per order) ---
reviews_dedup = (
    reviews.sort_values("review_answer_timestamp", na_position="last")
    .drop_duplicates("order_id", keep="last")
    [["order_id", "review_score"]]
)
print(f"reviews_dedup: {len(reviews_dedup):,} rows")

# --- Payments (primary = highest value per order) ---
payments_dedup = (
    payments.sort_values("payment_value", ascending=False)
    .drop_duplicates("order_id", keep="first")
    [["order_id", "payment_type", "payment_value"]]
)
print(f"payments_dedup: {len(payments_dedup):,} rows")

reviews_dedup: 98,673 rows
payments_dedup: 99,440 rows


In [8]:
# --- Final merge ---
master = (
    orders
    .merge(customers_geo, on="customer_id", how="left")
    .merge(items_agg, on="order_id", how="left")
    .merge(reviews_dedup, on="order_id", how="left")
    .merge(payments_dedup, on="order_id", how="left")
)
print(f"master (all Brazil): {master.shape[0]:,} rows × {master.shape[1]} cols")
print(f"Columns: {master.columns.tolist()}")

master (all Brazil): 99,441 rows × 28 cols
Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'customer_lat', 'customer_lon', 'price', 'freight_value', 'product_weight_g', 'n_items', 'product_id', 'product_category_name_english', 'seller_lat', 'seller_lon', 'seller_state', 'seller_city', 'seller_zip_code_prefix', 'review_score', 'payment_type', 'payment_value']


## 5. Feature engineering

**is_return** definition (per session summary):
- `order_status ∈ {canceled, unavailable}`, **OR**
- `actual_delivery > estimated_delivery + 7 days`

In [9]:
# Parse timestamps
for col in ["order_purchase_timestamp", "order_delivered_customer_date",
            "order_estimated_delivery_date"]:
    master[col] = pd.to_datetime(master[col], errors="coerce")

# delivery_days: purchase → actual delivery
master["delivery_days"] = (
    master["order_delivered_customer_date"] - master["order_purchase_timestamp"]
).dt.days

# days_late: actual − estimated (positive = late)
master["days_late"] = (
    master["order_delivered_customer_date"] - master["order_estimated_delivery_date"]
).dt.days

# order_value
master["order_value"] = master["price"].fillna(0) + master["freight_value"].fillna(0)

# --- is_return ---
RETURN_STATUSES = {"canceled", "unavailable"}
LATE_THRESHOLD = 7

status_return = master["order_status"].isin(RETURN_STATUSES)
late_return   = master["days_late"] > LATE_THRESHOLD
master["is_return"] = (status_return | late_return.fillna(False)).astype(int)

print(f"is_return breakdown:")
print(f"  status-based (canceled/unavailable): {status_return.sum():,}")
print(f"  late-delivery (>{LATE_THRESHOLD}d):            {late_return.sum():,}")
print(f"  combined (deduplicated):             {master['is_return'].sum():,}")
print(f"  return rate:                         {master['is_return'].mean()*100:.2f}%")

# --- return_rate_by_category ---
cat_rates = master.groupby("product_category_name_english")["is_return"].mean()
master["return_rate_by_category"] = master["product_category_name_english"].map(cat_rates)

# --- Fill NaN product_weight_g ---
median_wt = master["product_weight_g"].median()
master["product_weight_g"] = master["product_weight_g"].fillna(median_wt)
print(f"\nMedian product weight (fill value): {median_wt:.0f} g")

is_return breakdown:
  status-based (canceled/unavailable): 1,234
  late-delivery (>7d):            2,863
  combined (deduplicated):             4,096
  return rate:                         4.12%

Median product weight (fill value): 750 g


## 6. Filter to São Paulo (SP)

In [10]:
print(f"State distribution (top 10):")
print(master["customer_state"].value_counts().head(10).to_string())

sp = master[master["customer_state"] == "SP"].copy()
print(f"\nSP filter: {len(master):,} → {len(sp):,} rows ({len(sp)/len(master)*100:.1f}%)")

# Drop rows with missing coordinates
before = len(sp)
sp = sp.dropna(subset=["customer_lat", "customer_lon"])
print(f"Dropped {before - len(sp):,} rows with missing customer coords → {len(sp):,} final")

State distribution (top 10):
customer_state
SP    41746
RJ    12852
MG    11635
RS     5466
PR     5045
SC     3637
BA     3380
DF     2140
ES     2033
GO     2020

SP filter: 99,441 → 41,746 rows (42.0%)
Dropped 15 rows with missing customer coords → 41,731 final


## 7. Validation

In [11]:
print("=" * 60)
print("  MASTER_DF SUMMARY (SP only)")
print("=" * 60)
print(f"  Shape:             {sp.shape[0]:,} rows × {sp.shape[1]} cols")
print(f"  Unique orders:     {sp['order_id'].nunique():,}")
print(f"  Unique customers:  {sp['customer_id'].nunique():,}")
print(f"  Date range:        {sp['order_purchase_timestamp'].min().date()} → "
      f"{sp['order_purchase_timestamp'].max().date()}")
print(f"  Lat range:         [{sp['customer_lat'].min():.4f}, {sp['customer_lat'].max():.4f}]")
print(f"  Lon range:         [{sp['customer_lon'].min():.4f}, {sp['customer_lon'].max():.4f}]")
print(f"  Return rate:       {sp['is_return'].mean()*100:.2f}%")
print(f"  Avg delivery_days: {sp['delivery_days'].mean():.1f}")
print(f"  Avg days_late:     {sp['days_late'].mean():.1f}")
print(f"  Avg order_value:   R$ {sp['order_value'].mean():.2f}")
print(f"  Null counts:")
for col in ["customer_lat", "customer_lon", "seller_lat", "seller_lon",
            "review_score", "delivery_days", "product_weight_g"]:
    print(f"    {col:>35s}: {sp[col].isna().sum():,}")
print("=" * 60)

  MASTER_DF SUMMARY (SP only)
  Shape:             41,731 rows × 33 cols
  Unique orders:     41,731
  Unique customers:  41,731
  Date range:        2016-09-13 → 2018-10-17
  Lat range:         [-25.0102, -19.9427]
  Lon range:         [-53.0562, -44.3200]
  Return rate:       2.81%
  Avg delivery_days: 8.3
  Avg days_late:     -11.1
  Avg order_value:   R$ 141.85
  Null counts:
                           customer_lat: 0
                           customer_lon: 0
                             seller_lat: 448
                             seller_lon: 448
                           review_score: 274
                          delivery_days: 1,250
                       product_weight_g: 0


## 8. Save `master_df.parquet`

In [12]:
output_path = Path("../data/master_df.parquet")
output_path.parent.mkdir(parents=True, exist_ok=True)
sp.to_parquet(output_path, index=False)
print(f"✅ Saved {len(sp):,} rows → {output_path}")
print(f"   File size: {output_path.stat().st_size / 1e6:.1f} MB")

✅ Saved 41,731 rows → ..\data\master_df.parquet
   File size: 8.0 MB


## 9. Quick sanity check — reload and verify

In [14]:
check = pd.read_parquet("../data/master_df.parquet")
assert check.shape[0] == sp.shape[0], "Row count mismatch!"
assert "is_return" in check.columns, "is_return column missing!"
assert "days_late" in check.columns, "days_late column missing!"
assert "product_weight_g" in check.columns, "product_weight_g column missing!"
assert check["customer_state"].unique().tolist() == ["SP"], "Non-SP rows found!"
print("✅ All assertions passed. master_df.parquet is ready.")
print(f"   Columns: {check.columns.tolist()}")

✅ All assertions passed. master_df.parquet is ready.
   Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'customer_lat', 'customer_lon', 'price', 'freight_value', 'product_weight_g', 'n_items', 'product_id', 'product_category_name_english', 'seller_lat', 'seller_lon', 'seller_state', 'seller_city', 'seller_zip_code_prefix', 'review_score', 'payment_type', 'payment_value', 'delivery_days', 'days_late', 'order_value', 'is_return', 'return_rate_by_category']


## 10. Columns available for downstream modules

| Column | Used by | Notes |
|--------|---------|-------|
| `customer_lat/lon` | clustering, haversine_matrix, VRP | SP coordinates |
| `customer_zip_code_prefix` | clustering (demand weighting) | Group by zip for demand |
| `product_weight_g` | VRP capacity constraints | Sum per order, NaN filled |
| `is_return` | return_classifier (target), VRP | 0/1 binary |
| `days_late` | return_classifier (feature) | Positive = late |
| `review_score` | return_classifier (feature) | 1–5 ordinal |
| `freight_value` | return_classifier (feature) | R$ |
| `price` / `order_value` | return_classifier, KPIs | R$ |
| `product_category_name_english` | return_classifier, analysis | Translated category |
| `seller_state` | return_classifier (feature) | State code |
| `payment_type` | return_classifier (feature) | credit_card, etc. |
| `return_rate_by_category` | analysis, report | Category-level return % |
| `order_purchase_timestamp` | Prophet forecasting | Weekly aggregation |